In [0]:
# Demo 16 Setup: Scheduling, Refreshing and Subscribing
# Creates data, a dashboard, and PUBLISHES it so students can practice scheduling.

import re
import json
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType
from datetime import date, timedelta
import random

# --- Schema setup ---
username = spark.sql("SELECT current_user()").collect()[0][0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog = "main"
schema = f"demo_scheduling_{clean_username}"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
spark.sql(f"USE CATALOG `{catalog}`")
spark.sql(f"USE SCHEMA `{schema}`")

print(f"Schema: {catalog}.{schema}")

In [0]:
# --- gold_sales: 2000 rows ---
random.seed(55)

spark.sql("DROP TABLE IF EXISTS gold_sales")

regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'Northwest']
categories = ['Electronics', 'Clothing', 'Home & Garden', 'Sports', 'Food & Beverage']
segments = ['Enterprise', 'Mid-Market', 'SMB', 'Consumer']
channels = ['Online', 'Retail Store', 'Partner', 'Mobile App']

schema_def = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_date", DateType(), False),
    StructField("region", StringType(), False),
    StructField("product_category", StringType(), False),
    StructField("customer_segment", StringType(), False),
    StructField("channel", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("net_revenue", DoubleType(), False),
    StructField("cost", DoubleType(), False)
])

rows = []
for i in range(1, 2001):
    order_date = date(2024, 1, 1) + timedelta(days=random.randint(0, 364))
    unit_price = round(random.uniform(15, 500), 2)
    qty = random.randint(1, 6)
    discount = random.choice([0.0, 0.0, 0.0, 0.05, 0.10, 0.15, 0.20])
    net_rev = round(qty * unit_price * (1 - discount), 2)
    cost_val = round(net_rev * random.uniform(0.4, 0.7), 2)
    rows.append((
        i, order_date,
        random.choice(regions), random.choice(categories),
        random.choice(segments), random.choice(channels),
        qty, net_rev, cost_val
    ))

df = spark.createDataFrame(rows, schema=schema_def)
df.write.saveAsTable("gold_sales")
print(f"Created gold_sales: {df.count()} rows")

In [0]:
# --- Create AND Publish a dashboard ---
# Scheduling requires a PUBLISHED dashboard, so we create one and publish it.

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
parent_path = f"/Users/{username}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

table_ref = f"`{catalog}`.`{schema}`.`gold_sales`"

# Define datasets
datasets = [
    {
        "name": "ds_kpi",
        "displayName": "KPI Summary",
        "query": f"SELECT ROUND(SUM(net_revenue), 0) AS total_revenue, COUNT(order_id) AS total_orders, ROUND(SUM(net_revenue) - SUM(cost), 0) AS total_profit FROM {table_ref}"
    },
    {
        "name": "ds_region",
        "displayName": "Revenue by Region",
        "query": f"SELECT region, ROUND(SUM(net_revenue), 2) AS total_revenue, COUNT(order_id) AS order_count FROM {table_ref} GROUP BY region ORDER BY total_revenue DESC"
    },
    {
        "name": "ds_monthly",
        "displayName": "Monthly Trend",
        "query": f"SELECT DATE_TRUNC('month', order_date) AS order_month, ROUND(SUM(net_revenue), 2) AS total_revenue FROM {table_ref} GROUP BY DATE_TRUNC('month', order_date) ORDER BY order_month"
    }
]

# Define page with widgets
page = {
    "name": "page_main",
    "displayName": "Sales Overview",
    "layout": [
        {
            "widget": {
                "name": "w_title",
                "textbox_spec": "# Sales Dashboard\n\nScheduled refresh demo. This dashboard is PUBLISHED and ready for scheduling."
            },
            "position": {"x": 0, "y": 0, "width": 6, "height": 1}
        },
        {
            "widget": {
                "name": "w_revenue",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi", "fields": [{"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "counter", "encodings": {"value": {"fieldName": "total_revenue", "displayName": "Total Revenue"}}}
            },
            "position": {"x": 0, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_orders",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi", "fields": [{"name": "total_orders", "expression": "`total_orders`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "counter", "encodings": {"value": {"fieldName": "total_orders", "displayName": "Total Orders"}}}
            },
            "position": {"x": 2, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_profit",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_kpi", "fields": [{"name": "total_profit", "expression": "`total_profit`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "counter", "encodings": {"value": {"fieldName": "total_profit", "displayName": "Total Profit"}}}
            },
            "position": {"x": 4, "y": 1, "width": 2, "height": 2}
        },
        {
            "widget": {
                "name": "w_bar",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_region", "fields": [{"name": "region", "expression": "`region`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "bar", "encodings": {"x": {"fieldName": "region", "scale": {"type": "categorical"}, "displayName": "Region"}, "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Revenue"}}}
            },
            "position": {"x": 0, "y": 3, "width": 3, "height": 3}
        },
        {
            "widget": {
                "name": "w_line",
                "queries": [{"name": "main_query", "query": {"datasetName": "ds_monthly", "fields": [{"name": "order_month", "expression": "`order_month`"}, {"name": "total_revenue", "expression": "`total_revenue`"}], "disaggregated": True}}],
                "spec": {"version": 3, "widgetType": "line", "encodings": {"x": {"fieldName": "order_month", "scale": {"type": "temporal"}, "displayName": "Month"}, "y": {"fieldName": "total_revenue", "scale": {"type": "quantitative"}, "displayName": "Revenue"}}}
            },
            "position": {"x": 3, "y": 3, "width": 3, "height": 3}
        }
    ]
}

dashboard_definition = {"pages": [page], "datasets": datasets}
serialized = json.dumps(dashboard_definition)
dashboard_name = f"Sales Dashboard - Scheduling Demo ({clean_username})"

# Step 1: Create the dashboard
resp = requests.post(
    f"https://{workspace_url}/api/2.0/lakeview/dashboards",
    headers=headers,
    json={"display_name": dashboard_name, "serialized_dashboard": serialized, "parent_path": parent_path}
)

if resp.status_code == 200:
    dashboard_id = resp.json()["dashboard_id"]
    print(f"\u2705 Dashboard created: {dashboard_name}")
    print(f"   ID: {dashboard_id}")
else:
    # Dashboard may already exist - try to find it
    print(f"\u26a0\ufe0f  Create returned {resp.status_code}: {resp.text[:200]}")
    print("   Attempting to find existing dashboard...")
    list_resp = requests.get(
        f"https://{workspace_url}/api/2.0/lakeview/dashboards",
        headers=headers,
        params={"page_size": 100}
    )
    dashboard_id = None
    if list_resp.status_code == 200:
        for d in list_resp.json().get("dashboards", []):
            if d.get("display_name") == dashboard_name:
                dashboard_id = d["dashboard_id"]
                print(f"   Found existing dashboard: {dashboard_id}")
                break
    if not dashboard_id:
        print("   Could not find dashboard. Please check your workspace.")

# Step 2: Publish the dashboard
if dashboard_id:
    pub_resp = requests.post(
        f"https://{workspace_url}/api/2.0/lakeview/dashboards/{dashboard_id}/published",
        headers=headers,
        json={"embed_credentials": True}
    )
    if pub_resp.status_code == 200:
        print(f"\u2705 Dashboard PUBLISHED successfully!")
    else:
        print(f"\u26a0\ufe0f  Publish returned {pub_resp.status_code}: {pub_resp.text[:200]}")

In [0]:
# --- Summary ---
print("="*60)
print("SETUP COMPLETE")
print("="*60)
print(f"Catalog:  {catalog}")
print(f"Schema:   {schema}")
print(f"")
print(f"Table created:")
print(f"  gold_sales - 2000 rows")
print(f"")
print(f"Dashboard created AND published:")
print(f"  Name: {dashboard_name}")
print(f"  Status: PUBLISHED (ready for scheduling)")
print(f"  Contents: 3 datasets, 6 widgets")
print(f"")
print(f"Your task in the demo: Set up scheduled refreshes and subscriptions.")
print(f"")
print(f"To find your dashboard:")
print(f"  1. Click 'Dashboards' in the left sidebar")
print(f"  2. Look for: {dashboard_name}")
print(f"  3. Open the PUBLISHED version")